In [ ]:
import random
import json
import os
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import make_llm_request, adapt_model_kwargs_for_model
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS
from vpei.epistemic_consistency.experiment_types import carry_out_unblind_experiment

In [ ]:
models = ["gpt-5-mini"]
n = 2
custom_model_kwargs = {}
path_to_save_model_outputs = "./unblind_experiment"
random_seed = 41
experiment_name = "evaluate_policy_effectiveness_given_contingency_tables"

In [ ]:
# Load the data. political_pole is already in the CSV (left or right).
df = pd.read_csv("./data/contingency_tables.csv")

# The unblinded prompt reveals the political pole of the policy.
system_prompt = EXPERIMENTS[experiment_name]["unblind_experiment"]["system_prompt"]
user_prompt_template = EXPERIMENTS[experiment_name]["unblind_experiment"]["user_prompt_template"]
variables = ["problem", "a", "b", "c", "d", "political_pole"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(user_prompt_template)

In [ ]:
random.seed(1)  # for reproducibility

# test request
model_name = models[0]
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})
user_prompt = user_prompt_template.format(**{var: df.loc[0, var] for var in variables})
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
response = make_llm_request(model_name, messages, **model_kwargs)
print(response)

In [ ]:
payloads = await carry_out_unblind_experiment(
    models=models,
    df=df,
    variables=variables,
    n=n,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
    custom_model_kwargs=custom_model_kwargs,
    path_to_save_model_outputs=path_to_save_model_outputs,
    random_seed=random_seed,
)
df_results = pd.DataFrame(payloads)

In [ ]:
df_results.groupby(["model_name", "political_pole"])["model_response"].mean().reset_index()